In [19]:
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3, boxscoreadvancedv3
import pandas as pd
import time

In [20]:
game_log = leaguegamelog.LeagueGameLog(
    season="2025-26",
    season_type_all_star="Regular Season",
)

games_df = game_log.get_data_frames()[0]

In [21]:
game_ids = games_df["GAME_ID"].drop_duplicates().tolist()

In [22]:
len(game_ids)

1230

In [23]:
test_game_ids = game_ids[:10]
test_game_ids

['0022500001',
 '0022500002',
 '0022500089',
 '0022500080',
 '0022500003',
 '0022500081',
 '0022500083',
 '0022500088',
 '0022500004',
 '0022500082']

In [24]:
player_game_dfs = []
advanced_game_dfs = []

for game_id in test_game_ids:
    box = boxscoretraditionalv3.BoxScoreTraditionalV3(
        game_id=game_id
    )
    advanced_box = boxscoreadvancedv3.BoxScoreAdvancedV3(
        game_id=game_id
    )

    advanced_player_df = advanced_box.get_data_frames()[0]
    player_df = box.get_data_frames()[0]

    player_game_dfs.append(player_df)
    advanced_game_dfs.append(advanced_player_df)

    time.sleep(0.6)

In [25]:
player_games_df = pd.concat(
    player_game_dfs,
    ignore_index=True
)
advanced_games_df = pd.concat(
    advanced_game_dfs,
    ignore_index=True
)

In [26]:
player_games_df.shape

(272, 34)

In [27]:
advanced_games_df.shape

(272, 37)

In [28]:
player_games_df.head()

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,...,reboundsOffensive,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints
0,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1629652,Luguentz,Dort,L. Dort,...,2,4,6,5,1,0,1,4,6,2.0
1,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1631096,Chet,Holmgren,C. Holmgren,...,2,5,7,2,0,0,3,6,28,-1.0
2,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628392,Isaiah,Hartenstein,I. Hartenstein,...,5,3,8,5,2,0,1,6,6,9.0
3,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1641717,Cason,Wallace,C. Wallace,...,1,6,7,5,4,0,2,3,14,8.0
4,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628983,Shai,Gilgeous-Alexander,S. Gilgeous-Alexander,...,0,5,5,5,2,2,3,2,35,3.0


In [29]:
game_dates = (
    games_df[["GAME_ID", "GAME_DATE"]]
    .drop_duplicates()
    .rename(columns={"GAME_ID": "gameId", "GAME_DATE": "gameDate"})
)

In [30]:
player_games_df = player_games_df.merge(
    game_dates,
    on="gameId",
    how="left"
)

In [31]:
player_games_df.head()

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,...,reboundsDefensive,reboundsTotal,assists,steals,blocks,turnovers,foulsPersonal,points,plusMinusPoints,gameDate
0,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1629652,Luguentz,Dort,L. Dort,...,4,6,5,1,0,1,4,6,2.0,2025-10-21
1,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1631096,Chet,Holmgren,C. Holmgren,...,5,7,2,0,0,3,6,28,-1.0,2025-10-21
2,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628392,Isaiah,Hartenstein,I. Hartenstein,...,3,8,5,2,0,1,6,6,9.0,2025-10-21
3,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1641717,Cason,Wallace,C. Wallace,...,6,7,5,4,0,2,3,14,8.0,2025-10-21
4,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628983,Shai,Gilgeous-Alexander,S. Gilgeous-Alexander,...,5,5,5,2,2,3,2,35,3.0,2025-10-21


In [32]:
player_games_df["gameDate"] = pd.to_datetime(
    player_games_df["gameDate"]
)

In [33]:
advanced_features = [
    "gameId",
    "personId",
    "teamId",
    "teamTricode",
    "minutes",

    # offense / efficiency
    "effectiveFieldGoalPercentage",
    "trueShootingPercentage",
    "usagePercentage",

    # playmaking
    "assistPercentage",
    "assistToTurnover",
    "assistRatio",
    "turnoverRatio",

    # rebounding
    "offensiveReboundPercentage",
    "defensiveReboundPercentage",
    "reboundPercentage",

    # impact / defense proxy
    "offensiveRating",
    "defensiveRating",
    "netRating",

    # context / workload
    "possessions",
    "PIE",
]

In [34]:
advanced_player_clean = advanced_games_df[advanced_features]

In [35]:
player_full_df = player_games_df.merge(
    advanced_player_clean,
    on=["gameId", "personId"],
    how="left",
    suffixes=("", "_advanced")
)

In [36]:
player_full_df.head()

,gameId,teamId,teamCity,teamName,teamTricode,teamSlug,personId,firstName,familyName,nameI,...,assistRatio,turnoverRatio,offensiveReboundPercentage,defensiveReboundPercentage,reboundPercentage,offensiveRating,defensiveRating,netRating,possessions,PIE
0,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1629652,Luguentz,Dort,L. Dort,...,26.3,5.3,0.038,0.089,0.062,111.9,104.5,7.4,84.0,0.015
1,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1631096,Chet,Holmgren,C. Holmgren,...,8.7,13.0,0.054,0.161,0.103,122.7,119.2,3.4,75.0,0.139
2,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628392,Isaiah,Hartenstein,I. Hartenstein,...,41.7,8.3,0.125,0.079,0.103,121.7,102.7,19.0,69.0,0.071
3,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1641717,Cason,Wallace,C. Wallace,...,27.8,11.1,0.023,0.143,0.081,119.2,104.9,14.3,78.0,0.156
4,0022500001,1610612760,Oklahoma City,Thunder,OKC,thunder,1628983,Shai,Gilgeous-Alexander,S. Gilgeous-Alexander,...,12.2,7.3,0.000,0.109,0.051,111.1,104.3,6.8,90.0,0.182


In [37]:
player_full_df["trueShootingPercentage"].isna().mean()

np.float64(0.0)

In [38]:
player_full_df.duplicated(
    subset=["gameId", "personId"]
).sum()

np.int64(0)

In [39]:
player_full_df["gameId"].nunique()

10

In [40]:
player_full_df.shape

(272, 53)

In [41]:
player_full_df.to_csv(
    "../data/processed/player_games.csv",
    index=False
)

In [42]:
import pandas as pd

traditional_df = pd.read_csv(
    "../data/raw/traditional_player_games.csv"
)

advanced_df = pd.read_csv(
    "../data/raw/advanced_player_games.csv"
)

In [43]:
print("Traditional rows:", len(traditional_df))
print("Advanced rows:", len(advanced_df))

print(
    "Traditional games:",
    traditional_df["gameId"].nunique()
)

print(
    "Advanced games:",
    advanced_df["gameId"].nunique()
)

Traditional rows: 32515
Advanced rows: 32713
Traditional games: 1230
Advanced games: 1230


In [44]:
traditional_games = set(
    traditional_df["gameId"].astype(str)
)

advanced_games = set(
    advanced_df["gameId"].astype(str)
)

print(
    "Missing in advanced:",
    traditional_games - advanced_games
)

print(
    "Missing in traditional:",
    advanced_games - traditional_games
)

Missing in advanced: set()
Missing in traditional: set()


In [45]:
print(
    "Traditional duplicates:",
    traditional_df.duplicated(
        subset=["gameId", "personId"]
    ).sum()
)

print(
    "Advanced duplicates:",
    advanced_df.duplicated(
        subset=["gameId", "personId"]
    ).sum()
)

Traditional duplicates: 0
Advanced duplicates: 198


In [46]:
advanced_df.duplicated().sum()

np.int64(0)

In [47]:
advanced_duplicates = advanced_df[
    advanced_df.duplicated(
        subset=["gameId", "personId"],
        keep=False
    )
].sort_values(
    ["gameId", "personId"]
)

advanced_duplicates[
    [
        "gameId",
        "personId",
        "firstName",
        "familyName",
        "minutes",
        "trueShootingPercentage",
        "usagePercentage",
        "offensiveRating",
        "defensiveRating",
        "netRating",
        "PIE",
    ]
].head(30)

,gameId,personId,firstName,familyName,minutes,trueShootingPercentage,usagePercentage,offensiveRating,defensiveRating,netRating,PIE
4302,22400005,1626171,Bobby,Portis Jr.,NaN,0.000,0.000,0.0,0.0,0.0,0.000
4303,22400005,1626171,Bobby,Portis,18:00,0.500,0.409,88.6,66.7,21.9,0.297
5009,22400012,202685,Jonas,Valanciunas,NaN,0.000,0.000,0.0,0.0,0.0,0.000
5010,22400012,202685,Jonas,Valančiūnas,16:53,0.510,0.178,78.9,133.3,-54.4,0.052
4803,22400017,1628427,Vlatko,Cancar,NaN,0.000,0.000,0.0,0.0,0.0,0.000
4804,22400017,1628427,Vlatko,Čančar,8:03,0.500,0.111,106.7,107.1,-0.5,0.122
4812,22400017,1630527,Brandon,Boston Jr.,NaN,0.000,0.000,0.0,0.0,0.0,0.000
4813,22400017,1630527,Brandon,Boston,37:54,0.684,0.193,106.9,106.8,0.1,0.124
5790,22400023,1628427,Vlatko,Cancar,NaN,0.000,0.000,0.0,0.0,0.0,0.000
5791,22400023,1628427,Vlatko,Čančar,11:23,0.833,0.125,100.0,82.1,17.9,0.149


In [48]:
advanced_duplicates.groupby(
    ["gameId", "personId"]
).size().value_counts()

2    198
Name: count, dtype: int64

In [49]:
sample_game_id = advanced_duplicates.iloc[0]["gameId"]
sample_person_id = advanced_duplicates.iloc[0]["personId"]

advanced_duplicates[
    (advanced_duplicates["gameId"] == sample_game_id)
    &
    (advanced_duplicates["personId"] == sample_person_id)
].T

,4302,4303
gameId,22400005,22400005
teamId,1610612749,1610612749
teamCity,Milwaukee,Milwaukee
teamName,Bucks,Bucks
teamTricode,MIL,MIL
teamSlug,bucks,bucks
personId,1626171,1626171
firstName,Bobby,Bobby
familyName,Portis Jr.,Portis
nameI,B. Portis Jr.,B. Portis


In [50]:
duplicate_check = (
    advanced_duplicates
    .assign(has_minutes=advanced_duplicates["minutes"].notna())
    .groupby(["gameId", "personId"])["has_minutes"]
    .agg(["sum", "count"])
)

duplicate_check.value_counts()

sum  count
1    2        198
Name: count, dtype: int64

In [51]:
advanced_df["has_minutes"] = (
    advanced_df["minutes"].notna()
)

advanced_df = (
    advanced_df
    .sort_values("has_minutes")
    .drop_duplicates(
        subset=["gameId", "personId"],
        keep="last"
    )
    .drop(columns="has_minutes")
    .reset_index(drop=True)
)

In [52]:
print("Traditional rows:", len(traditional_df))
print("Advanced rows:", len(advanced_df))

print(
    "Advanced duplicates:",
    advanced_df.duplicated(
        subset=["gameId", "personId"]
    ).sum()
)

Traditional rows: 32515
Advanced rows: 32515
Advanced duplicates: 0


In [53]:
traditional_keys = set(
    zip(
        traditional_df["gameId"],
        traditional_df["personId"]
    )
)

advanced_keys = set(
    zip(
        advanced_df["gameId"],
        advanced_df["personId"]
    )
)

print(
    "Missing player-games in advanced:",
    len(traditional_keys - advanced_keys)
)

print(
    "Missing player-games in traditional:",
    len(advanced_keys - traditional_keys)
)

Missing player-games in advanced: 0
Missing player-games in traditional: 0
